# Amazon Review 2023
Amazon Reviews dataset is large-scale dataset collected in 2023 by McAuley Lab, and it includes rich features such as:
- User Reviews (ratings, text, helpfulness votes, etc.);
- Item Metadata (descriptions, price, raw image, etc.);
- Links (user-item / bought together graphs).

Related Information
- HP: https://amazon-reviews-2023.github.io/
- paper: [Bridging Language and Items for Retrieval and Recommendation](https://arxiv.org/abs/2403.03952)


In [1]:
%load_ext autoreload

In [2]:
%autoreload 2

import pathlib

from torch_geometric.data import HeteroData

from ml_sandbox_libs.data.amazon_reviews_dataset import (
    AmazonReviewsSeqRecDataModule,
    bipartite_graph_preprocess_dataset,
    fetch_dataset,
    fetch_metadata,
)

/Users/haru256/repo/github.com/haru-256/ml-sandbox/libs/ml_sandbox_libs/.venv/lib/python3.12/site-packages/torch_geometric/typing.py:68: UserWarning: An issue occurred while importing 'pyg-lib'. Disabling its usage. Stacktrace: dlopen(/Users/haru256/repo/github.com/haru-256/ml-sandbox/libs/ml_sandbox_libs/.venv/lib/python3.12/site-packages/libpyg.so, 0x0006): Library not loaded: /Library/Frameworks/Python.framework/Versions/3.12/Python
  Referenced from: <441E30E4-F1D4-325A-924A-8C4E5BD0FA29> /Users/haru256/repo/github.com/haru-256/ml-sandbox/libs/ml_sandbox_libs/.venv/lib/python3.12/site-packages/libpyg.so
  Reason: tried: '/Library/Frameworks/Python.framework/Versions/3.12/Python' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/Library/Frameworks/Python.framework/Versions/3.12/Python' (no such file), '/Library/Frameworks/Python.framework/Versions/3.12/Python' (no such file)
  warnings.warn(f"An issue occurred while importing 'pyg-lib'. "
/Users/haru256/repo/github.com/haru-256/

In [3]:
dataset_dict = fetch_dataset(category="Video_Games", dataset_type="0core_timestamp_w_his")
df = dataset_dict["train"].to_polars()
df.head()

2025-09-08 23:35:50.344 | INFO     | ml_sandbox_libs.data.amazon_reviews_dataset.common:fetch_dataset:41 - Fetching Amazon Reviews 2023 dataset


user_id,parent_asin,rating,timestamp,history
str,str,str,str,str
"""AGCI7FAH4GL5FI65HYLKWTMFZ2CQ""","""B07SRWRH5D""","""5.0""","""1587051114941""",""""""
"""AGCI7FAH4GL5FI65HYLKWTMFZ2CQ""","""B07DK1H3H5""","""4.0""","""1608186804795""","""B07SRWRH5D"""
"""AGXVBIUFLFGMVLATYXHJYL4A5Q7Q""","""B07MFMFW34""","""5.0""","""1490877431000""",""""""
"""AFTC6ZR5IKNRDG5JCPVNVMU3XV2Q""","""B00HUWA45W""","""5.0""","""1427591932000""",""""""
"""AFTC6ZR5IKNRDG5JCPVNVMU3XV2Q""","""B0BCHWZX95""","""5.0""","""1577637634017""","""B00HUWA45W"""


In [4]:
print("Dataset Size")
print(
    f"train: {len(dataset_dict['train'])}, valid: {len(dataset_dict['valid'])}, test: {len(dataset_dict['test'])}"
)

Dataset Size
train: 3847041, valid: 344592, test: 363867


The dataset schema is as follows: https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023#for-user-reviews

| Field            | Type     | Explanation                                                                                                                                                                               |
|------------------|----------|-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| user_id          | str      | ID of the reviewer                                                                                                                                                                       |
| parent_asin      | str      | Parent ID of the product. Note: Products with different colors, styles, sizes usually belong to the same parent ID. The “asin” in previous Amazon datasets is actually parent ID. **Please use parent ID to find product meta.** |
| rating           | float    | Rating of the product (from 1.0 to 5.0).                                                                                                                                                   |
| timestamp        | int      | Time of the review (unix time)                                                                                                                                                           |
| history | str     | parent_asin list which was bought by user before. The separator is ' '                                                                                                                                                               |

In [5]:
metadata_dataset = fetch_metadata(category="Video_Games")
metadata_df = metadata_dataset.to_polars().select(
    ["parent_asin", "title", "categories", "main_category", "average_rating", "rating_number"]
)
metadata_df.head(5)

2025-09-08 23:35:53.217 | INFO     | ml_sandbox_libs.data.amazon_reviews_dataset.common:fetch_metadata:61 - Fetching Amazon Reviews 2023 metadata


parent_asin,title,categories,main_category,average_rating,rating_number
str,str,list[str],str,f64,i64
"""B000FH0MHO""","""Dash 8-300 Professional Add-On""","[""Video Games"", ""PC"", ""Games""]","""Video Games""",5.0,1
"""B00069EVOG""","""Phantasmagoria: A Puzzle of Fl…","[""Video Games"", ""PC"", ""Games""]","""Video Games""",4.1,18
"""B00Z9TLVK0""","""NBA 2K17 - Early Tip Off Editi…","[""Video Games"", ""PlayStation 4"", ""Games""]","""Video Games""",4.3,223
"""B07SZJZV88""","""Nintendo Selects: The Legend o…","[""Video Games"", ""Legacy Systems"", … ""Games""]","""Video Games""",4.9,22
"""B002WH4ZJG""","""Thrustmaster Elite Fitness Pac…","[""Video Games"", ""Legacy Systems"", … ""Fitness Accessories""]","""Video Games""",3.0,3


In [6]:
print(f"Parent Asin Size: {len(metadata_df)}")

Parent Asin Size: 137269


The metadata schema is as follows: https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023#for-item-metadata

| Field           | Type   | Explanation                                                                                                                                                             |
|-----------------|--------|-------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| parent_asin     | str    | Parent ID of the product.                                                                                                                                                |
| title           | str    | Name of the product.                                                                                                                                                     |
| categories      | list   | Hierarchical categories of the product.                                                                                                                                  |


In [8]:
datamodule.train_df.head(5)

user_id,user_index,parent_asin,item_index,category,category_index,average_rating,rating_number,rating,timestamp,history,history_index,history_category,history_category_index,history_average_rating,history_rating_number
str,i64,str,i64,str,i64,f64,i64,f64,i64,list[str],list[i64],list[str],list[i64],list[f64],list[i64]
"""AGCI7FAH4GL5FI65HYLKWTMFZ2CQ""",1216394,"""B07SRWRH5D""",30015,"""Video Games/PlayStation 4/Game…",164,4.8,9097,5.0,1587051114941,[],[],[],[],[],[]
"""AGCI7FAH4GL5FI65HYLKWTMFZ2CQ""",1216394,"""B07DK1H3H5""",27749,"""Video Games/PC/Games""",146,4.1,2015,4.0,1608186804795,"[""B07SRWRH5D""]",[30015],"[""Video Games/PlayStation 4/Games""]",[164],[4.8],[9097]
"""AGXVBIUFLFGMVLATYXHJYL4A5Q7Q""",0,"""B07MFMFW34""",29089,"""Video Games/PC/Games""",146,3.0,31,5.0,1490877431000,[],[],[],[],[],[]
"""AFTC6ZR5IKNRDG5JCPVNVMU3XV2Q""",961526,"""B00HUWA45W""",19297,"""Video Games/Xbox One/Accessori…",177,4.0,287,5.0,1427591932000,[],[],[],[],[],[]
"""AFTC6ZR5IKNRDG5JCPVNVMU3XV2Q""",961526,"""B0BCHWZX95""",33733,"""Video Games/Nintendo Switch/Ac…",121,4.6,19492,5.0,1577637634017,"[""B00HUWA45W""]",[19297],"[""Video Games/Xbox One/Accessories""]",[177],[4.0],[287]


In [7]:
datamodule = AmazonReviewsSeqRecDataModule(
    save_dir=pathlib.Path("../data"),
    batch_size=2,
    num_workers=4,
    max_seq_len=5,
    neg_sample_size=2,
    sampling_val_test=True,
    eval_negative_sample_size=10,
    filter_no_history=False,
)

datamodule.prepare_data()
datamodule.setup(stage="fit")
train_dataloader = datamodule.train_dataloader()
batch = next(iter(train_dataloader))

2025-09-08 23:35:54.465 | INFO     | ml_sandbox_libs.data.amazon_reviews_dataset.seq_rec:prepare_data:426 - Preprocessed dataset not found
2025-09-08 23:35:54.465 | INFO     | ml_sandbox_libs.data.amazon_reviews_dataset.common:fetch_dataset:41 - Fetching Amazon Reviews 2023 dataset
2025-09-08 23:35:55.538 | INFO     | ml_sandbox_libs.data.amazon_reviews_dataset.common:fetch_metadata:61 - Fetching Amazon Reviews 2023 metadata
2025-09-08 23:36:00.537 | INFO     | ml_sandbox_libs.data.amazon_reviews_dataset.seq_rec:seq_rec_preprocess_dataset:191 - Preprocessing the train dataset
2025-09-08 23:36:08.499 | INFO     | ml_sandbox_libs.data.amazon_reviews_dataset.seq_rec:seq_rec_preprocess_dataset:193 - Preprocessing the val dataset
2025-09-08 23:36:09.088 | INFO     | ml_sandbox_libs.data.amazon_reviews_dataset.seq_rec:seq_rec_preprocess_dataset:195 - Preprocessing the test dataset
/Users/haru256/repo/github.com/haru-256/ml-sandbox/libs/ml_sandbox_libs/.venv/lib/python3.12/site-packages/torch

ColumnNotFoundError: Caught ColumnNotFoundError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/Users/haru256/repo/github.com/haru-256/ml-sandbox/libs/ml_sandbox_libs/.venv/lib/python3.12/site-packages/torch/utils/data/_utils/worker.py", line 349, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "/Users/haru256/repo/github.com/haru-256/ml-sandbox/libs/ml_sandbox_libs/.venv/lib/python3.12/site-packages/torch/utils/data/_utils/fetch.py", line 52, in fetch
    data = [self.dataset[idx] for idx in possibly_batched_index]
            ~~~~~~~~~~~~^^^^^
  File "/Users/haru256/repo/github.com/haru-256/ml-sandbox/libs/ml_sandbox_libs/src/ml_sandbox_libs/data/amazon_reviews_dataset/seq_rec.py", line 343, in __getitem__
    neg_item_indexes, neg_category_indexes, neg_average_ratings = self.negative_sampling(
                                                                  ^^^^^^^^^^^^^^

In [ ]:
datamodule.train_df.head(5)

In [ ]:
batch.user_index, batch.pos_item_index, batch.neg_item_indexes, batch.item_history

## Bipartite Graph

In [ ]:
(
    all_df,
    user2index,
    item2index,
    category2index,
    item_index_2_category_index,
) = bipartite_graph_preprocess_dataset(dataset_dict=dataset_dict, metadata=metadata_dataset)

In [ ]:
all_df.head(5)

In [ ]:
import numpy as np
import polars as pl
import torch
import torch_geometric.transforms as T
from torch_geometric.loader import LinkNeighborLoader
from torch_geometric.sampler import NegativeSampling

user_index = torch.as_tensor(sorted(user2index.values()), dtype=torch.int64)

In [ ]:
item_df = pl.from_dict({"item_index": item2index.values()})
item2category_df = pl.from_dict(
    {
        "item_index": item_index_2_category_index.keys(),
        "category": item_index_2_category_index.values(),
    }
)
item_df = item_df.join(item2category_df, on="item_index", validate="m:1").sort("item_index")
item_index = item_df["item_index"].to_torch()
category_index = item_df["category"].to_torch()

In [ ]:
all_df.filter(pl.col("split") == "train")

In [ ]:
def create_bipartite_graph(
    split: str,
    all_df: pl.DataFrame,
    user2index: dict[str, int],
    item2index: dict[str, int],
    item_index_2_category_index: dict[int, int],
):
    user_index = torch.as_tensor(sorted(user2index.values()), dtype=torch.int64)
    item_df = pl.from_dict({"item_index": item2index.values()})
    item2category_df = pl.from_dict(
        {
            "item_index": item_index_2_category_index.keys(),
            "category": item_index_2_category_index.values(),
        }
    )
    item_df = item_df.join(item2category_df, on="item_index", validate="m:1").sort("item_index")
    item_index = item_df["item_index"].to_torch()
    category_index = item_df["category"].to_torch()

    match split:
        case "train":
            df = all_df.filter(pl.col("split") == "train")
        case "valid":
            df = all_df.filter(pl.col("split").is_in(["train", "valid"]))
        case "test":
            df = all_df.filter(pl.col("split").is_in(["train", "valid", "test"]))
        case _:
            raise ValueError(f"Invalid split: {split}")

    edge_index = torch.as_tensor(
        np.ascontiguousarray(df["user_index", "item_index"].to_numpy().T), dtype=torch.long
    )
    edge_label_index = torch.as_tensor(
        np.ascontiguousarray(
            df.filter(pl.col("split") == split)["user_index", "item_index"].to_numpy().T
        ),
        dtype=torch.long,
    )
    data = HeteroData(
        {
            "user": {"x": user_index.unsqueeze(-1), "user_index": user_index},
            "item": {
                "x": item_index.unsqueeze(-1),
                "item_index": item_index,
                "category_index": category_index,
            },
            ("user", "rates", "item"): {
                "edge_index": edge_index,
                "edge_label_index": edge_label_index,
            },
        }
    )
    return data

In [ ]:
train_data = create_bipartite_graph(
    split="valid",
    all_df=all_df,
    user2index=user2index,
    item2index=item2index,
    item_index_2_category_index=item_index_2_category_index,
)
val_data = create_bipartite_graph(
    split="valid",
    all_df=all_df,
    user2index=user2index,
    item2index=item2index,
    item_index_2_category_index=item_index_2_category_index,
)

transform = T.Compose([T.RemoveIsolatedNodes(), T.RemoveSelfLoops()])
train_data = transform(train_data)
val_data = transform(val_data)

In [ ]:
train_data

In [ ]:
neg_sampling = NegativeSampling(mode="triplet", amount=3)

loader = LinkNeighborLoader(
    data=train_data,
    num_neighbors=[10, 5],
    batch_size=2,
    edge_label_index=(
        ("user", "rates", "item"),
        train_data["user", "rates", "item"].edge_label_index,
    ),
    edge_label=None,
    neg_sampling=neg_sampling,
    shuffle=True,
)

In [ ]:
train_data["user", "rates", "item"].edge_index

In [ ]:
train_data.keys()

In [ ]:
val_data["user"].x.size()

In [ ]:
train_df = all_df.filter(pl.col("split") == "train")
user_item_edge_index = train_df["user_index", "item_index"].to_torch(dtype=pl.Int64)

data = HeteroData(
    {
        "user": {"x": user_index, "user_index": user_index},
        "item": {"x": user_index, "item_index": item_index, "category_index": category_index},
        ("user", "rates", "item"): {"edge_index": user_item_edge_index},
    }
)
transform = T.Compose([T.RemoveIsolatedNodes(), T.RemoveSelfLoops()])
data = transform(data)

In [ ]:
data = HeteroData(
    {
        "user": {"user_index": user_index},
        "item": {"item_index": item_index, "category_index": category_index},
        ("user", "rates", "item"): {"edge_index": user_item_edge_index},
    }
)

In [ ]:
data